# Section 3: Pooled training data and baselines

Builds the training data for the toxicity classifier (package 4) from four labelled sources and scores simple baselines on it.

Pipeline: load and map labels → cap each source → clean → remove duplicates → stratified train / validation split → baselines → leave-one-source-out check.

Toxicity definition :

> a comment is toxic if it insults, belittles, harasses, threatens or attacks a person or group (including third parties not in the discussion), or is clearly abusive or hostile. Negative emotion, strong disagreement, criticism of a workplace and swearing that is not aimed at anyone are not enough on their own.



The external labels are not rewritten to fit this definition. They follow the proposal's mapping below, so the sources still differ in what they call toxic (for example, Dota labels most swearing as toxic).

## 1. Setup

In [ ]:
# Add the repo root to Python's search path
import sys
sys.path.append('..')

from functions.pooled_data import prepare_pooled_data
from functions.baselines import run_baselines

import os
import json
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, PrecisionRecallDisplay

RANDOM_STATE = 42
CAP_PER_SOURCE = 10_000
TEST_SIZE = 0.2
THRESHOLD = 0.5
DATA_DIR = '../dataset'
OUT_DIR = '../dataset/pooled'
FIG_DIR = os.path.join(OUT_DIR, 'figures')

## 2. Load, clean and split

Map the four sources to binary labels according to the proposal.

| Source | File in dataset/ | Toxic (1) if |
|---|---|---|
| youtube | youtoxic_english_1000.csv | IsToxic is true |
| wikipedia | jigsaw-toxic-comment-classification-challenge/train.csv.zip | any of the six toxicity labels is 1 |
| news | jigsaw-unintended-bias-in-toxicity-classification/train.csv | target ≥ 0.5 |
| dota | gosu-ai-english-dota-chat/tagged-data.csv | target is 1 (mild) or 2 (strong) |

Cap Wikipedia and news at 10,000 rows each, preserving their class proportions;
retain all YouTube and Dota rows. Lower-case the text and apply cleanup_text.
Remove empty texts and duplicates across sources: keep one copy when labels
agree and remove all copies when they conflict. Then split 80/20, stratified
by source and label.

In [ ]:
train_df, val_df, audit = prepare_pooled_data(
    DATA_DIR, cap_per_source=CAP_PER_SOURCE, test_size=TEST_SIZE, random_state=RANDOM_STATE
)
pooled = pd.concat([train_df, val_df], ignore_index=True)

print('Removed rows:')
print(audit['removed'].groupby(['source', 'reason']).size().to_string())

stages = ['raw', 'sampled', 'cleaned', 'deduplicated', 'train', 'val']
audit['summary'].pivot(index='source', columns='stage', values='rows')[stages]

Cleaning and deduplication reduce 24,267 sampled comments to 23,984:
16 empty texts, 259 duplicate copies and 8 conflicting-label rows are removed.
The final split contains 19,187 training and 4,797 validation comments.

## 3. Save

Export the fixed splits for package 4 to dataset/pooled/, together with
the source counts and removal audit. Both CSVs contain orig_id, source,
text_raw, text and label. The manifest records settings and file hashes
for version checks. These exports are ignored by Git.

In [ ]:
os.makedirs(FIG_DIR, exist_ok=True)
train_df.to_csv(os.path.join(OUT_DIR, 'pooled_train.csv'), index=False)
val_df.to_csv(os.path.join(OUT_DIR, 'pooled_val.csv'), index=False)
audit['summary'].to_csv(os.path.join(OUT_DIR, 'data_summary.csv'), index=False)
audit['removed'].to_csv(os.path.join(OUT_DIR, 'removed_rows.csv'), index=False)

manifest = audit['manifest']
with open(os.path.join(OUT_DIR, 'manifest.json'), 'w', encoding='utf-8') as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)
print(f'Saved {len(train_df):,} training and {len(val_df):,} validation rows to {OUT_DIR}')

## 4. Explore the pooled data

Compare source sizes, class balance and comment lengths. The table reports
mean, median, p90, maximum word count and the proportion exceeding 80 words.

In [ ]:
lengths = pooled['text'].str.split().str.len()
counts = pooled.groupby('source')['label'].agg(rows='size', toxic='sum')

length_summary = lengths.groupby(pooled['source']).describe(percentiles=[0.5, 0.9])
length_summary = length_summary[['mean', '50%', '90%', 'max']].rename(columns={'50%': 'median', '90%': 'p90'})
length_summary['over_80_words'] = lengths.gt(80).groupby(pooled['source']).mean()
length_summary.to_csv(os.path.join(OUT_DIR, 'text_length_summary.csv'))
display(length_summary.round(2))

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].bar(counts.index, counts['rows'] - counts['toxic'], label='non-toxic')
ax[0].bar(counts.index, counts['toxic'], bottom=counts['rows'] - counts['toxic'], label='toxic')
ax[0].set(title='Rows per source', ylabel='Rows')
ax[0].legend()
ax[1].boxplot([lengths[pooled['source'] == s] for s in counts.index], tick_labels=counts.index, showfliers=False)
ax[1].set(title='Comment length (outliers hidden)', ylabel='Words')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'source_and_length.png'), dpi=160)
plt.show()

Dota messages have a median of 4 words, compared with 20 for YouTube and
37 for news and Wikipedia. About 23% of news, 24% of Wikipedia and 10% of
YouTube comments exceed 80 words; none of the retained Dota messages do.

## 5. Baselines

Train three baselines on pooled_train and evaluate new comments from the
same four sources in pooled_val, using a fixed 0.5 threshold.

- **Majority class:** assigns every comment the training toxic share as its score.
- **TF-IDF + logistic regression:** learns word unigrams and bigrams.
- **Balanced logistic regression:** gives more weight to minority-class errors.

Parameters are fixed in advance. Precision and recall concern toxic comments;
Macro F1 averages both classes; PR-AUC uses Average Precision (AP).
The helper also computes the transfer experiments reported in section 6.

In [ ]:
results = run_baselines(train_df, val_df, threshold=THRESHOLD, random_state=RANDOM_STATE)
for key, file in [('metrics', 'baseline_metrics.csv'), ('predictions', 'baseline_predictions.csv'),
                  ('loso_metrics', 'loso_metrics.csv'), ('loso_predictions', 'loso_predictions.csv')]:
    results[key].to_csv(os.path.join(OUT_DIR, file), index=False)

manifest['baselines'] = results['config']
with open(os.path.join(OUT_DIR, 'manifest.json'), 'w', encoding='utf-8') as f:
    json.dump(manifest, f, indent=2, ensure_ascii=False)

cols = ['n', 'prevalence', 'precision', 'recall', 'macro_f1', 'pr_auc']
metrics = results['metrics']
metrics[metrics['group'] == 'all'].set_index('model')[cols].round(3)

The majority baseline predicts no toxic comments, giving zero precision and
recall. Its AP of 0.152 equals the validation toxic share.
Class weighting raises recall from 0.387 to 0.694 but lowers precision from
0.934 to 0.685; AP remains similar at 0.768 and 0.767.

In [ ]:
# the same baselines per source
metrics[metrics['group'] != 'all'].set_index(['model', 'group'])[cols].round(3)

News recall increases from 7.5% to 36.3% with class weighting. Recall improves
on all four sources, at the cost of more false alarms. YouTube has only
198 validation comments, so small score differences may reflect sampling noise.

### Confusion matrices and precision-recall curves

The confusion matrices count errors at the fixed 0.5 threshold.
The PR curves show precision and recall across thresholds; the dashed
line marks the validation toxic share.

In [ ]:
predictions = results['predictions']
names = ['tfidf_logreg', 'tfidf_logreg_balanced']

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for a, name in zip(ax, names):
    rows = predictions[predictions['model'] == name]
    ConfusionMatrixDisplay.from_predictions(
        rows['label'], rows['predicted_label'], display_labels=['Non-toxic', 'Toxic'],
        cmap='Blues', ax=a, colorbar=False,
    )
    a.set_title(f'{name} @ {THRESHOLD}')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'baseline_confusion_matrices.png'), dpi=160)
plt.show()

fig, ax = plt.subplots(figsize=(7, 5))
for name in names:
    rows = predictions[predictions['model'] == name]
    PrecisionRecallDisplay.from_predictions(rows['label'], rows['proba'], name=name, ax=ax)
ax.axhline(val_df['label'].mean(), linestyle='--', color='grey', label='majority class')
ax.set_title('Precision-recall curves (validation)')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'baseline_pr_curves.png'), dpi=160)
plt.show()

Class weighting reduces missed toxic comments from 447 to 223, while
increasing false alarms from 20 to 233. The PR curves largely overlap,
consistent with their similar AP values.

## 6. Leave-one-source-out

Test transfer by excluding one source from training. Fit the balanced model
on the other three sources' training rows, then evaluate the excluded source's
validation rows.

Compare this with the pooled model from section 5 on the same comments.
The n_train column gives the transfer model's training size;
pr_auc_change is LOSO AP minus pooled AP.

In [ ]:
loso = results['loso_metrics'].set_index('held_out_source')
pooled_scores = metrics[metrics['model'] == 'tfidf_logreg_balanced'].set_index('group').loc[loso.index]

comparison = pd.DataFrame({
    'n_train': loso['n_train'],
    'macro_f1_pooled': pooled_scores['macro_f1'],
    'macro_f1_loso': loso['macro_f1'],
    'pr_auc_pooled': pooled_scores['pr_auc'],
    'pr_auc_loso': loso['pr_auc'],
})
comparison['pr_auc_change'] = comparison['pr_auc_loso'] - comparison['pr_auc_pooled']
comparison.to_csv(os.path.join(OUT_DIR, 'transfer_comparison.csv'))
comparison.round(3)

AP decreases for all four held-out sources, most for Wikipedia
(0.814 → 0.702) and news (0.376 → 0.280). The drops may reflect platform
differences as well as the smaller training sets.

## 7. Error analysis

Inspect mistakes from the pooled models in section 5. FN counts missed toxic
comments; FP counts non-toxic comments flagged as toxic.
The examples show the balanced model's most confident errors.
All errors from both models are saved in baseline_errors.csv.

In [ ]:
errors = predictions[predictions['model'].isin(names) & predictions['error_type'].isin(['FP', 'FN'])]
errors.to_csv(os.path.join(OUT_DIR, 'baseline_errors.csv'), index=False)
error_counts = errors.groupby(['model', 'source', 'error_type']).size().unstack(fill_value=0)
error_counts.to_csv(os.path.join(OUT_DIR, 'error_counts.csv'))
display(error_counts)

pd.set_option('display.max_colwidth', 150)
balanced = errors[errors['model'] == 'tfidf_logreg_balanced']
show = ['source', 'label', 'proba', 'text_raw']
print('Most confident false positives (predicted toxic, labelled non-toxic):')
display(balanced[balanced['error_type'] == 'FP'].nlargest(10, 'proba')[show])
print('Most confident false negatives (predicted non-toxic, labelled toxic):')
display(balanced[balanced['error_type'] == 'FN'].nsmallest(10, 'proba')[show])

High-confidence false positives include "i am white" and "wtf is this",
suggesting sensitivity to identity terms and non-directed profanity.
Both are labelled non-toxic in the source data.

## 8. Handoff to package 4

- Load pooled_train.csv and pooled_val.csv with keep_default_na=False and
  orig_id as a string. Use text as input and label as the target.
- Text is already lower-cased and cleaned. Apply the same pipeline to new
  comments, followed by the saved tokenizer and padding settings.
- Fit the tokenizer, run cross-validation and tune thresholds only within
  pooled_train. Reserve pooled_val and the final r/jobs test set for evaluation.
- Compare the Bi-LSTM and baselines on the same validation split using the
  shared precision, recall, Macro F1 and AP functions.
- Regenerate the split and retrain if data, mappings or cleaning change.
  Check split_sha256 in manifest.json to match data versions.